# 7. Full sweep and report

Every method on one shared corpus, then the comparison table.

Before running this on a real tier, check the cost. `plan` counts what is already cached
and projects the time without generating anything, which is how you avoid discovering a
quota overrun three hours in.

```
silentwall plan --config configs/default.yaml
```

In [ ]:
# On Kaggle or Colab, uncomment to install
# !pip install -q -e /kaggle/working/silentwall
# !pip install -q -e .

from silentwall.config import load_config
from silentwall.pipeline import prepare_workspace, run_method, run_sweep, save_workspace
from silentwall.report.render import render_comparison, render_markdown, write_comparison

CONFIG = "../configs/smoke.yaml"
cfg = load_config(CONFIG)
print(cfg.profile, cfg.tier, "methods:", len(cfg.methods))

In [ ]:
results = run_sweep(cfg)

## The table

Read the middle two columns together. Low leakage with high AUC is the failure mode this
benchmark exists to surface: the content is hidden and the barrier is not.

In [ ]:
print(render_comparison(results))

In [ ]:
write_comparison(results, "../outputs")
for r in results:
    print(r.method_id, "->", r.run.run_id)

## Reproducibility

Every run record carries the config hash, corpus hash, seeds, library versions, and the
split audit. A reader can recompute the dev and eval id hashes from the config and corpus
and verify that the reported numbers came from entities the defense never calibrated on.

In [ ]:
r = results[0].run
for key in ("config_hash", "corpus_hash", "dev_ids_hash", "eval_ids_hash", "splits_disjoint"):
    print(f"{key}: {getattr(r, key)}")
print("seeds:", dict(r.seeds))